## Structured output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

In [2]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001F713748D70>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001F7137497F0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [3]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str = Field(description = "The Title is the Movie")
    year:int = Field(description = "This yesr the movie was release")
    director:str = Field(description = "This is the director name")
    rating:float = Field(description = "The rating of the movie")

In [6]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

RunnableBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001F713748D70>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001F7137497F0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The Title is the Movie', 'type': 'string'}, 'year': {'description': 'This yesr the movie was release', 'type': 'integer'}, 'director': {'description': 'This is the director name', 'type': 'string'}, 'rating': {'description': 'The rating of the movie'

In [7]:
model.invoke("Provide details about the moview Inception")

AIMessage(content='<think>\nOkay, so I need to provide details about the movie Inception. Let me start by recalling what I know about it. Inception is a 2010 science fiction action film directed by Christopher Nolan. The main actor is Leonardo DiCaprio, who plays the lead character Dom Cobb. The title itself, Inception, has something to do with planting ideas into people\'s minds, right? So the movie is about a thief who enters people\'s dreams to steal secrets, but then he\'s given a chance to have his criminal history erased by performing the reverse: planting an idea into a target\'s subconscious. That\'s the basic premise.\n\nNow, I should outline the main plot points. The protagonist, Dom Cobb, is a professional thief who enters the dreams of others to steal information. His team is offered a chance to erase their criminal past by performing an inception, which is the opposite of extraction. The target is Robert Fischer, the son of a business tycoon named Maurice Fischer. The plan

In [9]:
responce = model_with_structure.invoke("Provide details about the moview Inception")
responce

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

## Message output alongside parsed structure

In [10]:
class Movie(BaseModel):
    """Movie with the details"""
    title: str = Field(description = "This is the movie title")
    year: int = Field(description = "Movie release year")
    director:str = Field(description = "Movie director name")
    rating:float = Field(description = "Movie rating")

model_with_structure = model.with_structured_output(Movie, include_raw=True)
responce = model_with_structure.invoke("Provide details about the movie Inception")
responce

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. Let me check what tools I have available. There\'s a Movie function that requires title, year, director, and rating. I need to make sure I have all these details for Inception.\n\nFirst, the title is obviously "Inception". The release year was 2010. The director is Christopher Nolan. As for the rating, I think it\'s around 8.8 on IMDb. Let me confirm that. Yep, 8.8 is correct. \n\nSo I need to structure the function call with those parameters. Make sure all required fields are included. No missing data here. Let me put that into the JSON format as specified. Double-check the types: title and director are strings, year is an integer, rating is a number. Everything looks good. Ready to output the tool call.\n', 'tool_calls': [{'id': '7bjxkwa1y', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'nam

## Nested Structure

In [11]:
class Actor(BaseModel):
    name: str
    role:str

class Movie(BaseModel):
    title:str
    year:int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(description = "Budget in millions USD")

model_with_structure = model.with_structured_output(Movie)
model_with_structure.invoke("Provide details about the movie Inception")

Movie(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne')], genres=['Action', 'Science Fiction', 'Thriller'], budget=160.0)

In [12]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie avengers")
response



{'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers', 'year': 2012}